In [2]:
from physics.simulation import mcfm, msq
from physics.analysis import zz4l
from physics.hstar import c6

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt

In [3]:
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "xelatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})
lw  = 1.5

In [4]:
filenames = {
    msq.Component.SIG : 'data/zz4l/ggZZ_sig/analyzed.csv',
    msq.Component.BKG : 'data/zz4l/ggZZ_bkg/analyzed.csv',
    msq.Component.INT : 'data/zz4l/ggZZ_int/analyzed.csv',
    msq.Component.SBI : 'data/zz4l/ggZZ_sbi/analyzed.csv',
    'qq' : 'data/zz4l/qqZZ/analyzed.csv',
}

In [5]:
events_sig = mcfm.from_csv(file_path = filenames[msq.Component.SIG], kinematics = ['4l_mass'])
events_bkg = mcfm.from_csv(file_path = filenames[msq.Component.BKG], kinematics = ['4l_mass'])
events_int = mcfm.from_csv(file_path = filenames[msq.Component.INT], kinematics = ['4l_mass'], ignore_negative_weights=False)
events_sbi = mcfm.from_csv(file_path=filenames[msq.Component.SBI], kinematics = ['4l_mass'])
events_qq  = mcfm.from_csv(file_path=filenames['qq'], kinematics = ['4l_mass'])

In [6]:
nxbins = 41
xmin, xmax = 180, 1000.0
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)

In [7]:
xobs_sig = events_sig.kinematics['4l_mass']
xobs_bkg = events_bkg.kinematics['4l_mass']
xobs_int = events_int.kinematics['4l_mass']
xobs_sbi = events_sbi.kinematics['4l_mass']
xobs_qq  = events_qq.kinematics['4l_mass']

h_sig = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sig.fill(xobs_sig, weight=events_sig.weights)

h_bkg = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_bkg.fill(xobs_bkg, weight=events_bkg.weights)

h_int = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_int.fill(xobs_int, weight=events_int.weights)

h_sbi = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sbi.fill(xobs_sbi, weight=events_sbi.weights)

h_qq = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_qq.fill(xobs_qq, weight=events_qq.weights)

Hist(Variable(array([ 180.,  200.,  220.,  240.,  260.,  280.,  300.,  320.,  340.,
        360.,  380.,  400.,  420.,  440.,  460.,  480.,  500.,  520.,
        540.,  560.,  580.,  600.,  620.,  640.,  660.,  680.,  700.,
        720.,  740.,  760.,  780.,  800.,  820.,  840.,  860.,  880.,
        900.,  920.,  940.,  960.,  980., 1000.]), label='Axis 0'), storage=Weight()) # Sum: WeightedSum(value=44.8129, variance=0.000998251) (WeightedSum(value=44.8716, variance=0.00100344) with flow)

In [9]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,4), sharex=True)

l_sig = ax1.stairs(h_sig.values()/xwidths, xbins, color='tomato',   linestyle='--', linewidth=lw)
l_bkg = ax1.stairs(h_bkg.values()/xwidths, xbins, color='lightgrey',  linestyle='--', linewidth=lw)
l_sbi = ax1.stairs(h_sbi.values()/xwidths, xbins, color='black', linestyle='-',  linewidth=lw)
l_qq  = ax1.stairs(h_qq.values()/xwidths,  xbins, color='orange', linestyle='-', linewidth=lw)

l_sig.set_label('$gg,\\ |\\mathcal{M}_{\\mathrm{s}}|^2$')
l_bkg.set_label('$gg,\\ |\\mathcal{M}_{\\mathrm{b}}|^2$')
l_sbi.set_label('$gg,\\ |\\mathcal{M}_{\\mathrm{s}} + \\mathcal{M}_{\\mathrm{b}}|^2$')
l_qq.set_label('$q\\bar{q}$')
ax1.legend(frameon=False, fontsize=12)

ax1.set_yscale('log')
ax1.set_ylabel('$d\\sigma / d m_{ZZ}\\ \\mathrm{[fb / GeV]}$', loc='top', fontsize=15)
ax1.set_ylim(1e-5, 2.0)
ax1.yaxis.set_ticks([ 1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0])
ax1.yaxis.set_ticklabels([ '', '$10^{-4}$', '$10^{-3}$', '$10^{-2}$', '$0.1$', '$1.0$'])
ax1.text(0.00 ,0.03, '$\\approx$', transform=ax1.transAxes, ha='center', va='center', fontsize=12, bbox={'facecolor':'white', 'edgecolor':'none', 'pad':0.0})

ax1.tick_params(axis='x', direction='in')

ax1.text(0.04 ,0.12, '$pp \\rightarrow ZZ$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)
ax1.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)

l_int = ax2.stairs(h_int.values()/xwidths, xbins, color='cornflowerblue',  linestyle='--', linewidth=lw)
l_int.set_label('$gg,\\ 2\\mathrm{Re}(\\mathcal{M}^{\\dag}_{\\mathrm{s}} \\mathcal{M}_{\\mathrm{b}})$')
ax2.legend(frameon=False, fontsize=12, loc='lower right')

ax2.set_yscale('linear')
# ax2.tick_params(top=True, labeltop=False)

ax2.set_xlim(xmin, xmax)
ax2.xaxis.set_ticks([ 180, 250, 500, 750, 1000 ])
ax2.set_xlabel('$m_{ZZ}\\ \\mathrm{[GeV]}$', loc='right', fontsize=15)

ax2.set_ylim(-0.004, 0.0)
ax2.yaxis.set_ticks([-0.004, -0.002, 0.0])
ax2.yaxis.set_ticklabels(['$-0.004$', '$-0.002$', '$0$'])

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)

fig.tight_layout()
fig.subplots_adjust(hspace=0.00)

fig.canvas.draw()  # update positions

plt.savefig('pp4l_mzz_sm.pdf', bbox_inches='tight')
plt.close()